# Stage B4 — Export for iOS (Core ML)

**Experiment B — Practical application: MobileCLIP → SigLIP 2 adapter**

Goal: one linear matrix that maps iPhone-tier MobileCLIP-S1 image
embeddings into server-tier SigLIP 2 space, so a **single Qdrant
collection** serves both tiers — the phone indexes images offline, the
server queries the same index with SigLIP text embeddings.


## What this stage does
Exports the trained adapter in two forms:
- `adapter_fp16.npz` (~1 MB) — universal fallback, e.g. for MLX
- `Adapter.mlpackage` — a Core ML model (MatMul + L2-normalize, fp16,
  iOS 16+) that runs on the Neural Engine

## iPhone pipeline after success
`image → MobileCLIP encoder (Apple's official Core ML release) →
Adapter.mlpackage → vector in SigLIP space → shared Qdrant collection`

In Xcode: drag the `.mlpackage` into the project; Swift auto-generates a
class with a single `prediction(mobileclip_embedding:)` call.


In [ ]:
# =====================================================================
# WHERE THIS RUNS AND WHERE DATA GOES - read before running
# =====================================================================
# RUNTIME (choose in the Colab menu, or just open this locally):
#   Colab  : Runtime -> Change runtime type -> T4 / L4 GPU. A free T4 is
#            enough for most notebooks here; CPU works but is slow.
#   Local  : open in Jupyter on a machine with a CUDA GPU or Apple
#            Silicon. Nothing needs changing - the google.colab import
#            below fails harmlessly and it falls through to local mode.
#            To drive a local kernel from the Colab UI: install
#            jupyter_http_over_ws, launch jupyter with
#            --NotebookApp.allow_origin='https://colab.research.google.com'
#            and paste the printed localhost URL (with its token) into
#            Connect -> Connect to a local runtime.
#
# STORAGE (set STORAGE below):
#   "drive" : Google Drive at MyDrive/convergence_experiment  [DEFAULT]
#             Survives session timeouts, so checkpoints resume. On a
#             local machine this falls back to LOCAL_DIR automatically.
#   "local" : LOCAL_DIR on whatever machine is running. On a hosted
#             Colab VM this disk is ERASED at session end - downloads
#             and checkpoints do not survive.
#   "env"   : whatever DATA_DIR is already set to in the environment.
#
# The same folder can be shared between Colab and a local machine (the
# caches are plain .npz) - point both at one synced Drive folder.
# =====================================================================
import os
from pathlib import Path

STORAGE   = "drive"                     # "drive" | "local" | "env"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"

try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR is unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - Drive unavailable, using LOCAL_DIR instead")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())

DATA_DIR = Path(os.environ["DATA_DIR"])
DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    import torch
    _dev = ("cuda" if torch.cuda.is_available()
            else "mps" if getattr(torch.backends, "mps", None)
            and torch.backends.mps.is_available() else "cpu")
    _name = torch.cuda.get_device_name(0) if _dev == "cuda" else _dev
except ImportError:
    _dev, _name = "?", "torch not installed yet - run the pip cell"
print(f"environment : {'Colab' if IN_COLAB else 'local machine'}")
print(f"device      : {_dev} ({_name})")
print(f"DATA_DIR    : {DATA_DIR}")
if _dev == "cpu":
    print("WARNING: no GPU detected - encoding will take hours, not "
          "minutes")

In [ ]:
# Install dependencies (once)
# !pip install coremltools torch

In [ ]:
# Configuration and imports
import numpy as np
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "."))
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Functions
def main():
    ad = np.load(str(DATA_DIR / "adapter.npz"))
    W = ad["W_ridge"]  # [d_mob, d_sig]

    # 1. fp16 npz (universal fallback, e.g. for MLX)
    np.savez_compressed(str(DATA_DIR / "adapter_fp16.npz"), W=W.astype(np.float16))
    print(f"adapter_fp16.npz: {W.astype(np.float16).nbytes / 1e6:.2f} MB "
          f"({W.shape[0]} -> {W.shape[1]})")

    # 2. Core ML: y = l2norm(x @ W)
    try:
        import torch
        import coremltools as ct

        class Adapter(torch.nn.Module):
            def __init__(self, W):
                super().__init__()
                self.lin = torch.nn.Linear(W.shape[0], W.shape[1],
                                           bias=False)
                self.lin.weight.data = torch.tensor(W.T,
                                                    dtype=torch.float32)

            def forward(self, x):
                y = self.lin(x)
                return torch.nn.functional.normalize(y, dim=-1)

        model = Adapter(W).eval()
        dummy = torch.zeros(1, W.shape[0])
        traced = torch.jit.trace(model, dummy)

        mlmodel = ct.convert(
            traced,
            inputs=[ct.TensorType(name="mobileclip_embedding",
                                  shape=(1, W.shape[0]))],
            outputs=[ct.TensorType(name="siglip_space_embedding")],
            compute_precision=ct.precision.FLOAT16,
            minimum_deployment_target=ct.target.iOS16,
        )
        mlmodel.save(str(DATA_DIR / "Adapter.mlpackage"))
        print("Adapter.mlpackage saved — drop it into your Xcode project.")
        print("Swift usage: let out = try adapter.prediction("
              "mobileclip_embedding: emb)")
    except ImportError as e:
        print(f"Skipped Core ML export ({e}) — adapter_fp16.npz is enough; "
              "run this step on a machine with coremltools installed")

In [ ]:
# Run the export (requires adapter.npz from stage B2)
main()